# Demand Forecasting & Inventory Optimization
## Section 1: Data Cleaning

Here we are going to load our raw supply chain data and clean it up. We need to check for missing values, fix the date formats, and get rid of duplicates so it's ready for forecasting.

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')


### 1. Initial Data Inspection
Let's see how big the dataset is and look at the first few rows to understand what we're dealing with.

In [2]:
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


Next, let's see if there are any missing values that we need to fix.

In [3]:
# Checking for missing values in the entire dataframe
missing_values = df.isnull().sum()
missing_values[missing_values > 0]

Customer Lname              8
Customer Zipcode            3
Order Zipcode          155679
Product Description    180519
dtype: int64

### 2. Picking the right columns
We have 53 columns, but we don't need all of them. For forecasting, we just need the dates, product info, quantities, and prices. Let's pull those out and rename them to make our lives easier.

In [4]:
columns_to_keep = [
    'order date (DateOrders)', 'Order Id', 'Product Name', 
    'Category Name', 'Order Item Quantity', 'Order Region',
    'Product Price', 'Order Item Product Price', 'Order Item Total',
    'shipping date (DateOrders)', 'Days for shipping (real)',
    'Days for shipment (scheduled)'
]

df_subset = df[columns_to_keep].copy()

df_subset = df_subset.rename(columns={
    'order date (DateOrders)': 'order_date',
    'Order Id': 'order_id',
    'Product Name': 'product_name',
    'Category Name': 'category',
    'Order Item Quantity': 'quantity_sold',
    'Order Region': 'region',
    'Product Price': 'unit_price',
    'shipping date (DateOrders)': 'shipping_date',
    'Days for shipping (real)': 'actual_lead_time',
    'Days for shipment (scheduled)': 'scheduled_lead_time'
})

print("Selected columns:")
print(df_subset.columns.tolist())

Selected columns:
['order_date', 'order_id', 'product_name', 'category', 'quantity_sold', 'region', 'unit_price', 'Order Item Product Price', 'Order Item Total', 'shipping_date', 'actual_lead_time', 'scheduled_lead_time']


### 3. Quick sanity check
Let's run a quick summary to make sure we don't have any weird impossible values, like negative prices or zero quantities.

In [5]:
df_subset.describe()

,order_id,quantity_sold,unit_price,Order Item Product Price,Order Item Total,actual_lead_time,scheduled_lead_time
count,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000
mean,36221.894903,2.127638,141.232550,141.232550,183.107609,3.497654,2.931847
std,21045.379569,1.453451,139.732492,139.732492,120.043670,1.623722,1.374449
min,1.000000,1.000000,9.990000,9.990000,7.490000,0.000000,0.000000
25%,18057.000000,1.000000,50.000000,50.000000,104.379997,2.000000,2.000000
50%,36140.000000,1.000000,59.990002,59.990002,163.990005,3.000000,4.000000
75%,54144.000000,3.000000,199.990005,199.990005,247.399994,5.000000,4.000000
max,77204.000000,5.000000,1999.989990,1999.989990,1939.989990,6.000000,4.000000


Looks good! Now let's convert the date columns into actual datetime formats, drop any missing stuff, and remove duplicate rows just to be safe.

In [6]:
# Convert dates
df_subset['order_date'] = pd.to_datetime(df_subset['order_date'])
df_subset['shipping_date'] = pd.to_datetime(df_subset['shipping_date'])

# Drop any missing values in our subset
df_subset = df_subset.dropna()

# Drop duplicates
initial_len = len(df_subset)
df_subset = df_subset.drop_duplicates()
print(f"Dropped {initial_len - len(df_subset)} duplicate rows.")

# Filter out impossible values just in case
df_subset = df_subset[(df_subset['quantity_sold'] > 0) & (df_subset['unit_price'] >= 0)]
print(f"Final dataset shape: {df_subset.shape}")

Dropped 0 duplicate rows.
Final dataset shape: (180519, 12)


### 4. Final check
Let's see how many months of data we actually have and how many unique products there are.

In [7]:
print(f"Date range: {df_subset['order_date'].min()} to {df_subset['order_date'].max()}")
print(f"Total Unique Products (SKUs): {df_subset['product_name'].nunique()}")
print(f"Total Unique Categories: {df_subset['category'].nunique()}")

Date range: 2015-01-01 00:00:00 to 2018-01-31 23:38:00
Total Unique Products (SKUs): 118
Total Unique Categories: 50


### 5. Save it
Data is clean! Let's save it to a new CSV file so we can use it in the next section.

In [8]:
df_subset.to_csv('clean_supply_chain_data.csv', index=False)
print("Data successfully saved to 'clean_supply_chain_data.csv'")

Data successfully saved to 'clean_supply_chain_data.csv'
